# Spotify Playlist Explorer

Phase 1 demo (Sprint C, final): authenticate, pick a playlist (or your
Liked Songs), run six analyzers (genres, release year, top artists,
popularity, duration, timeline) with coordinated colorblind palette and
inline coverage annotations, render plots, and optionally export to parquet.

Setup: load credentials from `.env`, set the seaborn theme, and build a
cached `SpotifyClient`. The cache lives at `.cache/api/` (7-day TTL) so
repeated notebook runs don't re-hit the API.

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import seaborn as sns
from spotify_project.cache import FileCache
from spotify_project.client import SpotifyClient
from spotify_project.analyzer import (
    PlaylistAnalyzer,
    GenreAnalyzer,
    YearAnalyzer,
    ArtistAnalyzer,
    PopularityAnalyzer,
    DurationAnalyzer,
    TimelineAnalyzer,
)

load_dotenv()
sns.set_theme(style="whitegrid")
cache = FileCache(root=Path(".cache") / "api")
client = SpotifyClient.from_env(cache=cache)

## 1. Confirm authentication

In [ ]:
user = client.current_user()
print(f"Hello, {user['display_name']} ({user['id']})")

## 2. List your playlists

Spotify renamed `tracks` → `items` in their Feb 2026 migration. The `items` field is only present on playlists you own or collaborate on; for followed playlists, `tracks` reads as 0.

In [ ]:
import pandas as pd
playlists = client.user_playlists()
summary = pd.DataFrame([
    {
        'id': p.get('id', ''),
        'name': p.get('name', '<unnamed>'),
        'tracks': (p.get('items') or {}).get('total', 0),
        'owner': p.get('owner', {}).get('display_name', ''),
    }
    for p in playlists
])
summary.head(20)

## 3. Pick a playlist and fetch it

Replace `PLAYLIST_ID` below with one of the IDs from the table above.

In [ ]:
PLAYLIST_ID = "3v8PWRLiPHGPY0oHgkoZvV"  # or "__liked__" for your saved tracks
playlist = (
    client.liked_songs() if PLAYLIST_ID == "__liked__" else client.playlist(PLAYLIST_ID)
)
print(f"{playlist.name}: {len(playlist.tracks)} tracks")

## 4. Build the PlaylistAnalyzer with all six analyzers

Tweak knobs here:
- `YearAnalyzer(bucket_size=10)` for decade buckets — set to `1` for per-year bars.
- `ArtistAnalyzer(primary_only=True)` to ignore collaborators.
- `TimelineAnalyzer(freq='Y')` for yearly buckets instead of monthly.

Each analyzer reports a small summary DataFrame. Numbers come from the
flattened track DataFrame produced by `PlaylistAnalyzer.from_playlist`.
Coverage attached to each summary surfaces in the chart titles below.

In [ ]:
analyzers = [
    GenreAnalyzer(top_n=15),
    YearAnalyzer(bucket_size=10),
    ArtistAnalyzer(top_n=15, primary_only=False),
    PopularityAnalyzer(bins=10),
    DurationAnalyzer(bins=20),
    TimelineAnalyzer(freq="M"),
]
analyzer = PlaylistAnalyzer.from_playlist(playlist, analyzers=analyzers)
results = analyzer.run_all()
for title, df in results.items():
    print(title)
    print(df.head(), end="\n\n")

## 5. Render plots

All six panels stack vertically, each in a distinct colorblind-safe color.
`GenreAnalyzer` shows a small grey band at the bottom whenever some
tracks have no genre data — a common case for editorial playlists.

In [ ]:
fig = plt.figure(figsize=(12, 24))
analyzer.plot_all(fig)
plt.show()

## 6. (Optional) Export the flattened track DataFrame to parquet

Useful for offline analysis in another tool, or for archiving the snapshot
you analyzed today.

The exported parquet preserves the full flattened track DataFrame (track
id/name, primary artist, all artists, album, release date, duration in
minutes, popularity, added_at, genres). Useful for archiving the
snapshot you analyzed today.

In [ ]:
EXPORT = False  # set True to write the file
if EXPORT:
    out = Path("exports") / f"{playlist.id}.parquet"
    out.parent.mkdir(parents=True, exist_ok=True)
    analyzer.to_parquet(out)
    print(f"wrote {out}")